# E01:
train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [40]:
words = open('../../../data/name.txt', 'r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [41]:
import torch

In [ ]:
# FIRST APPROACH: STATISTICAL ------------------->
N3 = torch.zeros((27, 27, 27), dtype=torch.int32)

In [43]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

for w in words:
    chs = ["."] + ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N3[ix1, ix2, ix3] += 1

In [44]:
# how to use N?
# N[1, 2, 3] would be the count of 'abc' in the training data set
#
# start from N3[0, 0]
# normally, we already know the first two character and want to pick the third
# we use P3[ix1, ix2] to multinomially pick the third

P3 = (N3 + 1).float()
P3 /= P3.sum(2, keepdim=True)

In [66]:
for i in range(5):
    out = []
    # begin from ".."
    ix1, ix2 = 0, 0
    while True:
        p = P3[ix1, ix2]
        ix3 = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix3])
        ix1, ix2 = ix2, ix3
        if ix3 == 0:
            break
    print("".join(out))

kin.
keyn.
ali.
ani.
erayew.


In [67]:
# loss function
log_likelihood = 0.0
n = 0
for w in words:
    chs = ["."] + ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        prob = P3[ix1, ix2, ix3]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-504653.)
nll=tensor(504653.)
2.2119739055633545


In [68]:
# SECOND APPROACH: (54, 27) NEURAL NETWORK ----------->
# now xs, ys are the two input chars, zs is the label (char to predict)
xs, ys, zs = [], [], []
for w in words:
    chs = ["."] + ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)
num = xs.nelement()

In [69]:
# initialize the network
# the input is now TWO characters. each is one-hot encoded into a
# length-27 vector, and we concatenate them into one length-54 input.
# the output layer still has 27 neurons (one per possible next char),
# so the weight matrix W has shape (54, 27).
#   (num, 54) @ (54, 27) -> (num, 27)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

In [70]:
import torch.nn.functional as F

In [ ]:
# gradient descent
niters = 100
for k in range(niters):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float()   # (num, 27) one-hot of 1st char
    yenc = F.one_hot(ys, num_classes=27).float()   # (num, 27) one-hot of 2nd char
    xyenc = torch.cat([xenc, yenc], dim=1)         # (num, 54) concatenated input
    logits = xyenc @ W   # (num, 27) interpreted as log-counts; predict log-counts
    counts = logits.exp()   # (num, 27) equivalent to N
    probs = counts / counts.sum(1, keepdim=True) # (num, 27) probabilities for next char
    # negative log-likelihood of the true next char (zs), plus L2 regularization
    # loss = -probs[torch.arange(num), zs].log().mean() + 0.01 * (W**2).mean()
    loss = -probs[torch.arange(num), zs].log().mean()
    if k == niters - 1:
        print(loss.item())

    # backward pass
    W.grad = None  # set to zero the gradient
    loss.backward()

    # update
    W.data += -50 * W.grad

2.4312703609466553


In [80]:
for i in range(5):
    out = []
    # begin from ".."
    ix1, ix2 = 0, 0
    while True:
        # logits for the next char given the pair (ix1, ix2): this is exactly
        # what one_hot(ix1, ix2) @ W computes -> the two rows ADD (not average)
        logits = W[ix1] + W[ix2 + 27]
        counts = logits.exp()             # exp -> positive counts
        p = counts / counts.sum()         # normalize -> probability distribution
        ix3 = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix3])
        ix1, ix2 = ix2, ix3
        if ix3 == 0:
            break
    print("".join(out))

rohah.
ralaemaa.
breh.
cweli.
om.


In [72]:
# THIRD APPROACH: (27, 27, 27) NEURAL NETWORK ----------->
#
# the second approach used a (54, 27) W: it one-hot-encodes the two input
# chars SEPARATELY and ADDS their rows, so it can only learn a *factored*
# model  log count(c3) = f(c1, c3) + g(c2, c3). it cannot capture pair-
# specific interactions, so its loss bottoms out around ~2.27.
#
# to mimic the FIRST (counting) approach exactly, we need one row of logits
# per (ix1, ix2) PAIR context -> a (27, 27, 27) tensor W3, where
# W3[ix1, ix2, :] are the 27 log-counts for the next char given that pair.
# this plays the same role as log(N3): exp -> counts, normalize -> P3.
# gradient descent drives W3 toward log(N3) (the MLE), so the loss
# approaches the counting model's ~2.09.
#
# reuses xs, ys (the two input chars), zs (label) and num from above.

In [73]:
# initialize the network
# W3[ix1, ix2, :] holds the 27 log-counts for the next char given the pair
# (ix1, ix2). there are 27*27 = 729 pair-contexts, each with its own 27
# outputs -> 19,683 parameters (vs only 1,458 for the factored (54,27) net).
g = torch.Generator().manual_seed(2147483647)
W3 = torch.randn((27, 27, 27), generator=g, requires_grad=True)

In [74]:
# gradient descent
niters = 100
for k in range(niters):

    # forward pass
    # W3[xs, ys] is advanced indexing: for every example i it plucks the row
    # W3[xs[i], ys[i], :], producing (num, 27) logits. this is the learned
    # analogue of looking up N3[ix1, ix2] in the first approach.
    logits = W3[xs, ys]                              # (num, 27) log-counts
    counts = logits.exp()                            # (num, 27) counts ~ N3
    probs = counts / counts.sum(1, keepdim=True)     # (num, 27) softmax -> P3
    # negative log-likelihood of the true next char (zs), plus light L2 reg
    # loss = -probs[torch.arange(num), zs].log().mean() + 0.001 * (W3**2).mean()
    loss = -probs[torch.arange(num), zs].log().mean()
    if k == niters - 1:
        print(loss.item())

    # backward pass
    W3.grad = None
    loss.backward()

    # update
    W3.data += -50 * W3.grad

2.510735511779785


In [85]:

for i in range(5):
    out = []
    # begin from ".."
    ix1, ix2 = 0, 0
    while True:
        logits = W3[ix1, ix2]
        counts = logits.exp()             # exp -> positive counts
        p = counts / counts.sum()         # normalize -> probability distribution
        ix3 = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix3])
        ix1, ix2 = ix2, ix3
        if ix3 == 0:
            break
    print("".join(out))

iqbxbjqxsle.
dknxvbe.
jeowyellia.
kaej.
svodcuqvoctxxiqmbwhilnvex.
